# Indic Synthetic-Speech Pipeline — Colab (T4) stage-by-stage runbook

Validate **one stage at a time** on the GPU, then commit and move on.

**The loop (per stage):** edit locally in VSCode → `git push` → here run `!git pull` → run the stage → inspect the manifest + listen to audio → fix & repeat → commit `"stage N validated"`.

**Why outputs go to Drive:** each stage reads the previous stage's manifest, and Colab runtimes disconnect. With `out_dir` on Drive (see `config.colab.yaml`) the manifests + audio survive restarts, so the per-stage sessions below can run in *separate* runtimes.

**transformers conflict:** IndicF5 pins `==4.49.0` but Gemma-3 needs `>=4.50`. We handle it by giving each session its own install and **restarting the runtime** between Sessions 1→2→3.

> ⚠️ Never hardcode an HF token in a committed cell. We read it via `getpass`. If you ever leaked one, revoke it at https://huggingface.co/settings/tokens .

## 0. One-time setup (run at the start of every session)
Mount Drive, clone-or-pull the repo, set the HF token. Set `REPO_URL` to your GitHub repo.

In [1]:
import os
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/rahulkolayikkath/synthetic-data-pipeline.git'  
%cd /content
if not os.path.exists('/content/synthetic-data-pipeline'):
    !git clone $REPO_URL
%cd /content/synthetic-data-pipeline
!git pull --ff-only
os.makedirs('/content/drive/MyDrive/indic_synth/out', exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content
/content/synthetic-data-pipeline
Already up to date.


In [2]:
# HF token (kept out of git). Needs: accepted Kathbath terms + accepted Gemma-3 license.
from getpass import getpass
from huggingface_hub import login
os.environ['HF_TOKEN'] = getpass('HF token: ')
login(os.environ['HF_TOKEN'])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


---
## Session 1 — §4.2 acquisition + §4.3 audio engineering
No transformers model; CPU is fine. Run §4.2, inspect, then §4.3, inspect.

In [4]:
!pip install -q pyarrow pandas fsspec huggingface_hub datasets soundfile soxr librosa pyyaml
!pip install -q -e .

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for indic-synth (pyproject.toml) ... done


In [5]:
# §4.2 — pull the gender-balanced reference voice bank from Kathbath
!python scripts/run.py --config config.colab.yaml --stages data_acquisition

14:29:50 INFO    run | Pipeline start | out_dir=/content/drive/MyDrive/indic_synth/out seed=1234
14:29:50 INFO    run | =========== stage: data_acquisition ===========
14:29:50 INFO    run | Acquisition config: {'source': 'hf', 'repo_id': 'ai4bharat/Kathbath', 'local_dir': 'fake_kathbath', 'languages': ['hindi', 'malayalam'], 'split': 'valid', 'speakers_per_language': 10, 'clips_per_speaker': 4, 'min_total_speakers': 20, 'gender_balance': True, 'ref_min_dur': 3.0, 'ref_max_dur': 15.0, 'seed': 1234, 'out_dir': '/content/drive/MyDrive/indic_synth/out', 'hf_token': True, 'max_retries': 4, 'retry_backoff': 2.0, 'force_catalog': False}
14:29:51 INFO    run | [hindi] cataloging 2 parquet file(s) (audio column skipped)
14:30:03 INFO    run | [hindi] valid-00000-of-00002.parquet -> 1576 rows
14:30:15 INFO    run | [hindi] valid-00001-of-00002.parquet -> 1575 rows
14:30:15 INFO    run | [malayalam] cataloging 2 parquet file(s) (audio column skipped)
14:30:22 INFO    run | [malayalam] valid-0000

In [8]:
# inspect + listen to a reference clip
OUT = '/content/drive/MyDrive/indic_synth/out'
!python scripts/inspect_manifest.py $OUT/reference_manifest.jsonl --n 3
import glob
from IPython.display import Audio, display
for w in sorted(glob.glob(f'{OUT}/ref_audio/*'))[:2]:
    print(w); display(Audio(w))


=== /content/drive/MyDrive/indic_synth/out/reference_manifest.jsonl ===
rows: 80

-- categorical --
  status           {'downloaded': 80}
  lang             {'hindi': 40, 'malayalam': 40}
  gender           {'male': 40, 'female': 40}

-- numeric (min / mean / max) --
  duration         3.135 / 7.443 / 13.816

distinct (lang, speaker): 20

-- 3 sample rows --
  {"ref_id": "hindi_spk934_778", "lang": "hindi", "speaker_id": 934, "gender": "male", "source_repo": "ai4bharat/Kathbath", "source_split": "valid", "row_index": 778, "fname": "844424933473550-934-m.m4a", "ref_text": "नाइट्रोजन यौगीकीकरण जैसे द्वारा मिट्टी में प्राकृतिक रूप से किया जाता है", "duration": 8.336, "local_audio_path": "ref_audio/hindi_spk934_778.flac", "status": "downloaded"}
  {"ref_id": "hindi_spk934_563", "lang": "hindi", "speaker_id": 934, "gender": "male", "source_repo": "ai4bharat/Kathbath", "source_split": "valid", "row_index": 563, "fname": "844424933541613-934-m.m4a", "ref_text": "काच शलाका एवं नली का निर्माण 

/content/drive/MyDrive/indic_synth/out/ref_audio/hindi_spk1179_1287.flac


In [9]:
# §4.3 — decode / resample to 24 kHz mono / normalize
!python scripts/run.py --config config.colab.yaml --stages audio_engineering
!python scripts/inspect_manifest.py $OUT/prepared_manifest.jsonl --n 3

14:39:05 INFO    run | Pipeline start | out_dir=/content/drive/MyDrive/indic_synth/out seed=1234
14:39:05 INFO    run | =========== stage: audio_engineering ===========
14:39:05 INFO    run | 80 downloaded clips, 0 already prepared, 80 to process
14:39:07 INFO    run | Prepare complete: {"stage": "audio_engineering", "elapsed_sec": 2.44, "target_sr": 24000, "norm": "peak", "trim": false, "prepared": 80, "failed": 0, "sr_in": {"16000": 80}, "flags": {"resampled_up": 80, "input_clipped": 9}, "backends": {"soundfile": 80}, "prepared_manifest": "/content/drive/MyDrive/indic_synth/out/prepared_manifest.jsonl"}
14:39:07 INFO    run | Pipeline done in 2.5s. Final dataset: /content/drive/MyDrive/indic_synth/out/dataset_manifest.jsonl

=== /content/drive/MyDrive/indic_synth/out/prepared_manifest.jsonl ===
rows: 80

-- categorical --
  status           {'prepared': 80}
  lang             {'hindi': 40, 'malayalam': 40}
  gender           {'male': 40, 'female': 40}
  resample_method  {'soxr_hq': 8

In [10]:
# verify a prepared clip really is 24 kHz mono, and listen
import soundfile as sf, glob
from IPython.display import Audio, display
for w in sorted(glob.glob(f'{OUT}/prepared_audio/*.wav'))[:2]:
    i = sf.info(w); print(w, i.samplerate, 'Hz', i.channels, 'ch'); display(Audio(w))

/content/drive/MyDrive/indic_synth/out/prepared_audio/hindi_spk1179_1075.wav 24000 Hz 1 ch


/content/drive/MyDrive/indic_synth/out/prepared_audio/hindi_spk1179_1287.wav 24000 Hz 1 ch


**✅ If §4.2/§4.3 look right** (20 speakers, gender-balanced, 24 kHz mono refs): commit `"stage 4.2/4.3 validated"` from VSCode.

**Watch-outs:** gated 403 → accept Kathbath terms on the hub. If no files are found, the real HF layout / language-folder names may differ from the `datasets/ai4bharat/Kathbath/<lang>/valid-*.parquet` glob in `src/indic_synth/data_acquisition/hf_io.py` — adjust `languages` in `config.colab.yaml` or the glob. Kathbath clips may be **m4a** (decoded via librosa/ffmpeg in `audio_engineering/prepare.py`).

---
## Session 2 — §4.4 sentence generation (Gemma-3)
**Restart the runtime first** (Runtime → Restart), then re-run section 0, then this.
Needs `transformers>=4.50` + a GPU.

In [6]:
import os
os.environ["HF_HOME"] = "/content/drive/MyDrive/indic_synth/hf_cache"

In [3]:
!pip install -q "transformers>=4.50" accelerate bitsandbytes sentence-transformers fasttext-wheel indic-num2words pyyaml
!pip install -q -e .

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 120.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.2/314.2 kB 32.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for indic-synth (pyproject.toml) ... done


In [7]:
!pip uninstall -y fasttext fasttext-wheel
!pip install -q fasttext-numpy2-wheel

Found existing installation: fasttext-wheel 0.9.2
Uninstalling fasttext-wheel-0.9.2:
  Successfully uninstalled fasttext-wheel-0.9.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 53.7 MB/s eta 0:00:0000:0100:01


In [ ]:
# §4.4 — grid-balanced, validated Indic sentences (checkpointed; safe to re-run)
!python scripts/run.py --config config.colab.yaml --stages sentence_generation

15:16:02 INFO    run | Pipeline start | out_dir=/content/drive/MyDrive/indic_synth/out seed=1234
15:16:02 INFO    run | =========== stage: sentence_generation ===========
processor_config.json: 100% 70.0/70.0 [00:00<00:00, 311kB/s]
chat_template.json: 100% 1.61k/1.61k [00:00<00:00, 4.13MB/s]
preprocessor_config.json: 100% 570/570 [00:00<00:00, 2.42MB/s]
config.json: 100% 916/916 [00:00<00:00, 1.85MB/s]
tokenizer_config.json: 100% 1.16M/1.16M [00:00<00:00, 16.9MB/s]
tokenizer.json: 100% 33.4M/33.4M [00:00<00:00, 55.5MB/s]
added_tokens.json: 100% 35.0/35.0 [00:00<00:00, 84.7kB/s]
special_tokens_map.json: 100% 662/662 [00:00<00:00, 3.89MB/s]
model.safetensors.index.json: 100% 109k/109k [00:00<00:00, 68.1MB/s]
Fetching 5 files: 100% 5/5 [03:57<00:00, 47.49s/it] 
Download complete: 100% 24.4G/24.4G [03:57<00:00, 102MB/s]                
Loading weights:   0% 1/1065 [00:13<4:00:25, 13.56s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_

In [ ]:
OUT = '/content/drive/MyDrive/indic_synth/out'
!python scripts/inspect_manifest.py $OUT/sentences.jsonl --n 5
import json
print(open(f'{OUT}/sentences_summary.json').read())

**✅ Check:** per-language / per-type / per-topic counts are balanced, QC yield is reasonable, and sampled sentences read fluently. Commit `"stage 4.4 validated"`.

**Watch-outs:** accept the **Gemma-3 license**; `bitsandbytes` 4-bit needs the GPU; `lid.176.bin` (~126 MB) downloads into `out_dir`; confirm `processor.apply_chat_template(...)` works for the installed transformers (`sentence_generation/models.py`).

---
## Session 3 — §4.5 TTS (IndicF5) + §4.6 QC
**Restart the runtime first**, re-run section 0, then this. IndicF5 from source + `transformers==4.49.0`.

In [ ]:
!pip install -q git+https://github.com/ai4bharat/IndicF5.git
!pip install -q "transformers==4.49.0" torch torchaudio soundfile speechbrain jiwer pyyaml
!pip install -q -e .

In [ ]:
# §4.5 — IndicF5 speaks each sentence in a same-language speaker's voice
!python scripts/run.py --config config.colab.yaml --stages tts_generation
OUT = '/content/drive/MyDrive/indic_synth/out'
!python scripts/inspect_manifest.py $OUT/tts_manifest.jsonl --n 3

In [ ]:
# listen to a few synthesized utterances
import glob
from IPython.display import Audio, display
for w in sorted(glob.glob(f'{OUT}/tts_audio/*.wav'))[:4]:
    print(w); display(Audio(w))

In [ ]:
# §4.6 — three QC gates -> final dataset_manifest.jsonl
!python scripts/run.py --config config.colab.yaml --stages quality_control
!python scripts/inspect_manifest.py $OUT/dataset_manifest.jsonl --n 3
import json
print(open(f'{OUT}/qc_summary.json').read())

In [ ]:
# sanity-check thresholds: listen to a couple of QC failures
from indic_synth.common.manifest import read_jsonl
from IPython.display import Audio, display
rows = read_jsonl(f'{OUT}/dataset_manifest.jsonl')
fails = [r for r in rows if not r['qc_passed']][:3]
print('pass:', sum(r['qc_passed'] for r in rows), '/', len(rows))
for r in fails:
    print(r['utt_id'], 'CER=', r.get('cer'), 'spk=', r.get('speaker_sim'), r['qc_reasons'])
    display(Audio(f"{OUT}/{r['audio_filepath']}"))

**✅ Done** when `dataset_manifest.jsonl` holds ~1000 validated utterances across 2 languages / 20 speakers with a sensible QC pass rate. Commit `"stage 4.5/4.6 validated"`.

**Watch-outs:** IndicF5 git build + `trust_remote_code`; confirm the `tts_model(text, ref_audio_path=..., ref_text=...)` call + int16→float32 handling in `tts_generation/models.py`; T4 VRAM for IndicF5. For QC: the `indic-conformer` `model(wav, lang_code, 'ctc')` signature and the **lang_code it expects** (map `hi`/`ml` if needed); speechbrain ECAPA load. If conformer needs a different transformers than 4.49, run §4.6 in its own session — the Drive manifests make that safe.